# ComfyUI no Colab — instalação sob demanda por workflow

O *código* do ComfyUI fica no disco local do Colab (`/content`, rápido).
Modelos, outputs, inputs, workflows e o cache de custom nodes ficam no Drive.

**Só o ComfyUI-Manager é instalado sempre.** Todo o resto depende de quais
workflows você marcar na Célula 3 — o notebook lê o JSON de cada workflow,
descobre os `class_type` usados e instala apenas os pacotes necessários.

Ordem: **1 → 2 → 3 → 4 → (5 opcional) → 6**

> **Se algo nao bater com o esperado, voce pode estar com uma copia em cache.**
> O Colab guarda o `.ipynb` aberto pelo GitHub. Para forcar a versao nova:
> feche a aba e reabra o link, ou use *Arquivo -> Reverter para a versao salva*.
> A Celula 6 imprime `v6-wf-inject` — se nao imprimir, esta desatualizada.


In [ ]:
#@title 1. Montar Drive + instalar ComfyUI (local) { display-mode: "form" }
import os, sys, subprocess, pathlib
from google.colab import drive

DRIVE_ROOT = '/content/drive'
DRIVE_DATA = '/content/drive/MyDrive/ComfyUI_Data'  #@param {type:"string"}
COMFY      = '/content/ComfyUI'
CKOUT      = '/content/ComfyUI_Colab'
REPO       = 'https://github.com/BloomRX/ComfyUI_Colab'
BRANCH     = 'arena/01a05a82-comfyui-collab'

if not os.path.ismount(DRIVE_ROOT):
    drive.mount(DRIVE_ROOT)

# repo de config primeiro (leve) — traz as barras de progresso
if os.path.exists(CKOUT):
    subprocess.run('git pull -q', shell=True, cwd=CKOUT, check=False)
else:
    subprocess.run(f'git clone -q --depth 1 -b {BRANCH} {REPO} {CKOUT}', shell=True, check=True)
sys.path.insert(0, f'{CKOUT}/config')
import nbui

steps = nbui.Steps(4, 'Preparando ambiente')

steps.step('ComfyUI')
if not os.path.exists(COMFY):
    nbui.run(f'git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY}',
             'clonando ComfyUI')
else:
    nbui.run('git pull', 'atualizando ComfyUI', cwd=COMFY)

steps.step('pastas no Drive')
MODEL_DIRS = ['checkpoints','loras','vae','clip','clip_vision','controlnet',
              'upscale_models','embeddings','unet','diffusion_models','ipadapter',
              'animatediff_models','animatediff_motion_lora','sams','style_models',
              'text_encoders','skintoken','trellis2','birefnet']
for d in MODEL_DIRS + ['ultralytics/bbox','ultralytics/segm']:
    pathlib.Path(f'{DRIVE_DATA}/models/{d}').mkdir(parents=True, exist_ok=True)
for d in ['output','input','user','workflows']:
    pathlib.Path(f'{DRIVE_DATA}/{d}').mkdir(parents=True, exist_ok=True)

steps.step('extra_model_paths')
y = 'drive:\n  base_path: ' + DRIVE_DATA + '/models/\n'
y += ''.join(f'  {d}: {d}\n' for d in MODEL_DIRS)
y += '  ultralytics_bbox: ultralytics/bbox\n  ultralytics_segm: ultralytics/segm\n'
open(f'{COMFY}/extra_model_paths.yaml','w').write(y)

steps.step('dependencias')
nbui.run('pip install -r requirements.txt', 'pip install requirements', cwd=COMFY)

CN = f'{COMFY}/custom_nodes'
if not os.path.exists(f'{CN}/ComfyUI-Manager'):
    nbui.run(f'git clone --depth 1 https://github.com/Comfy-Org/ComfyUI-Manager "{CN}/ComfyUI-Manager"',
             'clonando ComfyUI-Manager')
    nbui.run(f'pip install -r "{CN}/ComfyUI-Manager/requirements.txt"', 'deps do Manager')

steps.close()
print('Workflows em:', f'{DRIVE_DATA}/workflows', '| e no repo:', f'{CKOUT}/Workflows')


In [ ]:
#@title 2. Registry de nodes (mapa class_type -> repositorio) { display-mode: "form" }
import json, os, glob, sys, subprocess, urllib.request

DRIVE_DATA = '/content/drive/MyDrive/ComfyUI_Data'
CKOUT = '/content/ComfyUI_Colab'
subprocess.run('git pull -q', shell=True, cwd=CKOUT, check=False)
sys.path.insert(0, f'{CKOUT}/config')
import importlib, nbui; importlib.reload(nbui)

LOCAL = f'{DRIVE_DATA}/node_registry.json'
REGISTRY = json.load(open(LOCAL if os.path.exists(LOCAL) else f'{CKOUT}/config/node_registry.json'))

PACKS         = REGISTRY['packs']
CLASS_MAP     = REGISTRY['class_map']
NATIVE_IGNORE = set(REGISTRY.get('native_ignore', []))
WF_DIRS = [f'{CKOUT}/Workflows', f'{DRIVE_DATA}/workflows']
print(f'OK: {len(PACKS)} pacotes, {len(CLASS_MAP)} nodes mapeados.')
for d in WF_DIRS:
    n = len(glob.glob(f'{d}/**/*.json', recursive=True)) if os.path.isdir(d) else -1
    print('  %-45s %s' % (d, 'NAO EXISTE' if n < 0 else f'{n} workflow(s)'))


In [ ]:
#@title 3. Listar workflows e ver o que cada um precisa { display-mode: "form" }
#@markdown Roda e olha a lista numerada. A escolha e feita na Celula 4.
import json, os, glob, re, urllib.request

uuidpat = re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-')
AUTO_RESOLVE = True  #@param {type:"boolean"}

files = []
for d in WF_DIRS:
    files += glob.glob(f'{d}/**/*.json', recursive=True)
files = sorted(set(files))
print('Procurado em:', WF_DIRS)
print('Encontrados :', len(files), 'arquivo(s)\n')

def classes_of(path):
    """Extrai class_types. Desce em subgraphs (definitions.subgraphs),
    senao 20+ nodes ficam invisiveis e o pacote deles nao e instalado."""
    try: d = json.load(open(path, encoding='utf-8'))
    except Exception as e:
        print('  erro lendo', path, e); return set()
    out = set()
    def walk_nodes(lst):
        for n in lst or []:
            if isinstance(n, dict) and n.get('type'): out.add(n['type'])
    if isinstance(d, dict) and 'nodes' in d:
        walk_nodes(d['nodes'])
        for sg in (d.get('definitions', {}) or {}).get('subgraphs', []) or []:
            walk_nodes(sg.get('nodes'))
    elif isinstance(d, dict):
        for n in d.values():
            if isinstance(n, dict) and n.get('class_type'): out.add(n['class_type'])
    return out

EXTMAP = {}
if AUTO_RESOLVE:
    CANDS = [
      '/content/ComfyUI/custom_nodes/ComfyUI-Manager/extension-node-map.json',
      'https://raw.githubusercontent.com/Comfy-Org/ComfyUI-Manager/main/extension-node-map.json',
      'https://github.com/Comfy-Org/ComfyUI-Manager/raw/refs/heads/main/extension-node-map.json',
    ]
    for src in CANDS:
        try:
            if src.startswith('http'):
                req = urllib.request.Request(src, headers={'User-Agent': 'Mozilla/5.0'})
                raw = json.loads(urllib.request.urlopen(req, timeout=60).read())
            else:
                if not os.path.exists(src): continue
                raw = json.load(open(src, encoding='utf-8'))
            for repo, entry in raw.items():
                for cls in entry[0]:
                    EXTMAP.setdefault(cls, repo)
            print(f'Auto-resolve: {len(EXTMAP)} nodes conhecidos.\n'); break
        except Exception as e:
            print(' auto-resolve falhou:', str(e)[:60])

NEVER = {x.lower() for x in REGISTRY.get('never_install', [])}

def resolve(cls):
    if cls in CLASS_MAP:
        p = CLASS_MAP[cls]; return (p, PACKS.get(p), 'registry')
    if cls in EXTMAP:
        url = EXTMAP[cls].rstrip('/')
        if url.endswith('.git'): url = url[:-4]
        pack = url.rsplit('/', 1)[-1]
        # o mapa do Manager lista nodes do CORE apontando para o repo do ComfyUI.
        # Clonar isso criaria custom_nodes/ComfyUI e quebraria o import.
        if pack.lower() in NEVER or url.lower().endswith('comfyanonymous/comfyui'):
            return None
        return (pack, url, 'auto')
    for pre, pname in (REGISTRY.get('prefix_hints') or {}).items():
        if cls.startswith(pre) and pname in PACKS:
            return (pname, PACKS[pname], 'prefixo')
    # display name tipo "Lora Loader Stack (rgthree)" -> deduz o pack do sufixo
    m = re.search(r'\(([^)]+)\)\s*$', cls)
    if m:
        hint = m.group(1).strip().lower()
        for pname in list(CLASS_MAP.values()) + list(PACKS):
            if hint and hint in pname.lower():
                return (pname, PACKS.get(pname), 'sufixo')
    return None

WF = []   # lista global usada pela Celula 4
for f in files:
    cls = classes_of(f)
    need, auto, unk = {}, [], []
    for c in cls:
        if c in NATIVE_IGNORE or uuidpat.match(c): continue
        r = resolve(c)
        if r is None: unk.append(c)
        else:
            need[r[0]] = r[1]
            if r[2] == 'auto': auto.append(r[0])
    WF.append({'path': f, 'name': os.path.basename(f), 'need': need, 'unk': unk})

if not WF:
    print('!! Nenhum .json encontrado. Confira se a Celula 2 clonou o repo.')
else:
    print('=' * 70)
    for i, w in enumerate(WF, 1):
        print(f"[{i}] {w['name']}")
        print(f"    pacotes: {', '.join(sorted(w['need'])) or 'so nodes nativos'}")
        if w['unk']: print(f"    [!] sem fonte: {', '.join(sorted(w['unk'])[:5])}")
    print('=' * 70)
    print('\nAgora va na Celula 4 e escreva os numeros que quer. Ex: 1  ou  1,3')


In [ ]:
#@title 4. Instalar os custom nodes dos workflows escolhidos { display-mode: "form" }
#@markdown Numeros da lista da Celula 3, separados por virgula. `all` = todos.
SELECAO = "1"  #@param {type:"string"}

import os, subprocess, json

COMFY='/content/ComfyUI'; CN=f'{COMFY}/custom_nodes'
DRIVE_DATA='/content/drive/MyDrive/ComfyUI_Data'
EXTRAS = REGISTRY.get('pack_extras', {})

sel = SELECAO.strip().lower()
if sel in ('all', 'todos', '*'):
    chosen = list(WF)
else:
    idx = [int(x) for x in re.findall(r'\d+', sel)]
    chosen = [WF[i-1] for i in idx if 1 <= i <= len(WF)]

if not chosen:
    raise SystemExit('Nada selecionado. Escreva um numero valido, ex: 1')

need = {}
for w in chosen: need.update(w['need'])
unk = sorted({u for w in chosen for u in w['unk']})

print('Workflows:', [w['name'] for w in chosen])
print('Pacotes  :', sorted(need) or '(nenhum)')
if unk: print('SEM FONTE (use o Manager depois):', unk)
print()

def sh(c, label=None, **k):
    return nbui.run(c, label or c[:50], **k)

apt = {a for pack in need for a in EXTRAS.get(pack, {}).get('apt', [])}
if apt:
    sh('apt-get -qq update && apt-get -qq install -y ' + ' '.join(sorted(apt)),
       'apt: ' + ' '.join(sorted(apt)))

prog = nbui.Steps(len(need) or 1, 'Instalando pacotes')

NEVER = {x.lower() for x in REGISTRY.get('never_install', [])}
for bad in ('ComfyUI', 'comfyui'):
    bp = f'{CN}/{bad}'
    if os.path.isdir(bp) and not os.path.exists(f'{bp}/__init__.py'):
        import shutil as _s; _s.rmtree(bp, ignore_errors=True)
        print(f'removido {bp} (invalido: quebrava o import)')

for pack, url in sorted(need.items()):
    prog.step(pack)
    if pack.lower() in NEVER:
        print(f'  ignorado {pack} (faz parte do core, nao e custom node)'); continue
    if not url:
        print(f'!! {pack} sem URL — instale pelo Manager.'); continue
    dst = f'{CN}/{pack}'
    if os.path.exists(dst + '.disabled') and not os.path.exists(dst):
        os.rename(dst + '.disabled', dst); print(f'reativado {pack}')
    if not os.path.exists(dst):
        sh(f'git clone --depth 1 {url} "{dst}"', f'clone {pack}')
    if not os.path.exists(dst):
        print(f'!! falha ao clonar {pack}'); continue
    if os.path.exists(f'{dst}/requirements.txt'):
        sh(f'pip install -r "{dst}/requirements.txt"', f'deps {pack}')
    if os.path.exists(f'{dst}/install.py'):
        sh('python install.py', f'install.py {pack} (compila CUDA, demora)', cwd=dst)
    for m in EXTRAS.get(pack, {}).get('hf_models', []):
        out = f'{DRIVE_DATA}/models/{m["dest"]}'
        os.makedirs(out, exist_ok=True)
        for fn in m['files']:
            if not os.path.exists(f'{out}/{fn}'):
                nbui.download(f'{m["repo_url"]}/resolve/main/{fn}', f'{out}/{fn}', fn)
    if EXTRAS.get(pack, {}).get('note'):
        print('   nota:', EXTRAS[pack]['note'])

prog.close()

# ---------- injeta os workflows escolhidos na aba "Workflows" da UI ----------
# A UI le de <user-directory>/default/workflows/ . So o formato UI (com "nodes")
# aparece la; formato API nao e abrivel pela aba.
import shutil
# Escreve nos DOIS: local (que a Celula 6 serve) e Drive (persistencia).
WF_UI_DIRS = [f'{DRIVE_DATA}/user/default/workflows',
              '/content/comfy_user/default/workflows']
WF_UI = WF_UI_DIRS[0]
for _d in WF_UI_DIRS: os.makedirs(_d, exist_ok=True)
for w in chosen:
    try:
        d = json.load(open(w['path'], encoding='utf-8'))
    except Exception as e:
        print(f"  nao consegui ler {w['name']}: {e}"); continue
    if not (isinstance(d, dict) and 'nodes' in d):
        print(f"  {w['name']}: formato API — nao aparece na aba, pule (use Load)"); continue
    src_bytes = open(w['path'], 'rb').read()
    novos = []
    for _d in WF_UI_DIRS:                     # garante nos DOIS, sempre
        alvo = f"{_d}/{w['name']}"
        if os.path.exists(alvo) and open(alvo, 'rb').read() == src_bytes:
            continue
        try:
            shutil.copy2(w['path'], alvo); novos.append(_d)
        except Exception as e:
            print(f"  ! falhou copiar para {_d}: {str(e)[:50]}")
    print(f"  {'+' if novos else '='} {w['name']}"
          + ('' if novos else ' (ja estava la)'))

# id precisa ser UUID: a aba Workflows indexa por ele e ignora outros formatos
import uuid as _uuid, re as _re
_U = _re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$')
for _d in WF_UI_DIRS:
    for _f in os.listdir(_d):
        if not _f.endswith('.json'): continue
        _p = f'{_d}/{_f}'
        try:
            _w = json.load(open(_p, encoding='utf-8'))
        except Exception:
            continue
        if isinstance(_w, dict) and 'nodes' in _w:
            _i = _w.get('id')
            if not (isinstance(_i, str) and _U.match(_i)):
                _w['id'] = str(_uuid.uuid5(_uuid.NAMESPACE_URL, 'wf/' + _f))
                json.dump(_w, open(_p, 'w'), indent=2, ensure_ascii=False)
                print(f'  (id corrigido para UUID: {_f})')

# ---- remove da UI os workflows que ESTE repo injetou e que nao existem mais --
# Sem isto, um workflow apagado do repo continua no Drive e na sidebar. Ao
# tentar abri-lo o frontend nao acha o arquivo, a restauracao de abas quebra
# no meio e NENHUMA aba responde mais ao clique.
_MANIFEST = f'{DRIVE_DATA}/user/.injetados.json'
try:
    _antes = set(json.load(open(_MANIFEST)))
except Exception:
    _antes = set()
_repo_hoje = {os.path.basename(w['path']) for w in WF}   # tudo que o repo tem
_copiados  = {w['name'] for w in chosen}                # o que ESTA sessao copiou
# orfao = foi injetado um dia E nao existe mais no repo.
# Nao basta 'nao escolhido agora': isso apagaria workflow de outra sessao.
_orfaos = sorted(_antes - _repo_hoje)
for _f in _orfaos:
    for _d in WF_UI_DIRS:
        _alvo = f'{_d}/{_f}'
        if os.path.exists(_alvo):
            try:
                os.remove(_alvo); print(f'  - removido (nao existe mais no repo): {_f}')
            except Exception as e:
                print(f'  ! nao consegui remover {_f}: {str(e)[:40]}')
try:
    os.makedirs(os.path.dirname(_MANIFEST), exist_ok=True)
    json.dump(sorted((_antes | _copiados) & _repo_hoje), open(_MANIFEST, 'w'), indent=1)
except Exception:
    pass

_ui = sorted(os.listdir(WF_UI_DIRS[1]))       # o que a UI realmente le
print(f'\nWorkflows visiveis na aba: {_ui}')
_extra = [x for x in _ui if x not in [w['name'] for w in chosen]]
if _extra:
    print(f'  (obs: {len(_extra)} de sessoes anteriores continuam la — normal.')
    print('   A celula nao apaga workflows; remova pela UI se quiser.)')
print()

for d in sorted(os.listdir(CN)):
    pth = f'{CN}/{d}'
    if not os.path.isdir(pth) or d == '__pycache__': continue
    if d.endswith('.disabled'): continue      # ja desativado; nao vira .disabled.disabled
    if d == 'ComfyUI-Manager' or d in need: continue
    if d.lower() in NEVER: continue
    alvo = pth + '.disabled'
    if os.path.exists(alvo):                  # sobra de sessao anterior
        import shutil as _s2; _s2.rmtree(alvo, ignore_errors=True)
    os.rename(pth, alvo); print(f'desativado {d}')

# conserta nomes acumulados de versoes antigas (.disabled.disabled...)
for d in sorted(os.listdir(CN)):
    if '.disabled.disabled' in d:
        base = d.split('.disabled')[0] + '.disabled'
        try:
            if os.path.exists(f'{CN}/{base}'):
                import shutil as _s3; _s3.rmtree(f'{CN}/{d}', ignore_errors=True)
            else:
                os.rename(f'{CN}/{d}', f'{CN}/{base}')
            print(f'corrigido nome: {d} -> {base}')
        except Exception as e:
            print(f'  (nao consegui corrigir {d}: {e})')

ativos = [d for d in sorted(os.listdir(CN))
          if os.path.isdir(f'{CN}/{d}') and not d.endswith('.disabled')
          and d != '__pycache__']
print('\nAtivos:', ativos)
faltando = [p for p in need if p not in ativos and p.lower() not in NEVER]
if faltando:
    print('!! esperados mas NAO ativos:', faltando)
    print('   Rode esta celula de novo; se persistir, instale pelo Manager.')


In [ ]:
#@title 5. Baixar os modelos dos workflows escolhidos { display-mode: "form" }
#@markdown Tres camadas: registry curado -> base do ComfyUI-Manager (auto) -> aviso.
#@markdown Funciona com workflows novos sem eu precisar editar nada.
PULAR_OPCIONAIS = True  #@param {type:"boolean"}
SO_LISTAR       = False #@param {type:"boolean"}
URL_EXTRA = ''  #@param {type:"string"}
PASTA_EXTRA = 'checkpoints'  #@param ["checkpoints","loras","vae","text_encoders","diffusion_models","controlnet","upscale_models","unet","clip_vision","embeddings","ipadapter"]
#@markdown **Token do HuggingFace** (so para modelos *gated*/privados).
#@markdown NAO cole o token aqui. Guarde em: painel lateral > chave 🔑 (Secrets)
#@markdown > `+ Adicionar novo secret`, nome **HF_TOKEN**, e ligue o acesso ao notebook.

import os, re, json, subprocess, shutil, urllib.request

DRIVE_DATA='/content/drive/MyDrive/ComfyUI_Data'
WFM   = REGISTRY.get('workflow_models', {})
NOTES = REGISTRY.get('model_notes', {})

try: chosen
except NameError: raise SystemExit('Rode a Celula 4 antes desta.')

# ---------- 1. varre o workflow atras de nomes de arquivo de modelo ----------
EXT = r'\.(safetensors|ckpt|gguf|pt|pth|bin|onnx|sft)$'
def wanted_files(path):
    try: d = json.load(open(path, encoding='utf-8'))
    except Exception: return set()
    out = set()
    def walk(o):
        if isinstance(o, str):
            s = o.strip()
            if re.search(EXT, s, re.I) and not s.lower().startswith('http'):
                out.add(s.replace('\\', '/').split('/')[-1])
        elif isinstance(o, list):
            for x in o: walk(x)
        elif isinstance(o, dict):
            for v in o.values(): walk(v)
    walk(d)
    return out

# ---------- 2. base de modelos do Manager (auto-resolve) ----------
MODELDB = {}
CANDS = ['/content/ComfyUI/custom_nodes/ComfyUI-Manager/model-list.json',
         'https://raw.githubusercontent.com/Comfy-Org/ComfyUI-Manager/main/model-list.json',
         'https://github.com/Comfy-Org/ComfyUI-Manager/raw/refs/heads/main/model-list.json']
for src in CANDS:
    try:
        if src.startswith('http'):
            rq = urllib.request.Request(src, headers={'User-Agent':'Mozilla/5.0'})
            raw = json.loads(urllib.request.urlopen(rq, timeout=60).read())
        else:
            if not os.path.exists(src): continue
            raw = json.load(open(src, encoding='utf-8'))
        for m in raw.get('models', []):
            fn = (m.get('filename') or m.get('name') or '').strip()
            if fn and m.get('url'): MODELDB.setdefault(fn, m)
        print(f'Base de modelos do Manager: {len(MODELDB)} arquivos.\n'); break
    except Exception as e:
        print(' base de modelos falhou:', str(e)[:60])

TYPE2DIR = {'checkpoints':'checkpoints','checkpoint':'checkpoints','lora':'loras','loras':'loras',
            'VAE':'vae','vae':'vae','controlnet':'controlnet','ControlNet':'controlnet',
            'upscale':'upscale_models','clip':'clip','clip_vision':'clip_vision',
            'unet':'unet','diffusion_model':'diffusion_models','text_encoders':'text_encoders',
            'embeddings':'embeddings','TAESD':'vae_approx'}

def already_have(fname):
    base = f'{DRIVE_DATA}/models'
    for root, _, files in os.walk(base):
        if fname in files and os.path.getsize(f'{root}/{fname}') > 1_000_000:
            return f'{root}/{fname}'.replace(base+'/', '')
    return None

# ---------- token do HuggingFace (nunca escrito no notebook) ----------
def _get_hf_token():
    """1) Colab Secrets  2) variavel de ambiente  3) pergunta escondido."""
    try:
        from google.colab import userdata
        t = (userdata.get('HF_TOKEN') or '').strip()
        if t: return t, 'Colab Secrets'
    except Exception:
        pass
    t = os.environ.get('HF_TOKEN', '').strip()
    if t: return t, 'variavel de ambiente'
    return '', ''

_tok, _src = _get_hf_token()
if _tok:
    print(f'HF_TOKEN carregado de: {_src} (…{_tok[-4:]})')
    os.environ['HF_TOKEN'] = _tok
else:
    print('HF_TOKEN nao configurado — so modelos publicos serao baixados.')

HDRS = {'Authorization': f'Bearer {_tok}'} if _tok else {}

def pedir_token():
    """Chame numa celula nova se preferir digitar o token na hora."""
    import getpass
    global _tok, HDRS
    _tok = getpass.getpass('Cole o token (nao aparece na tela): ').strip()
    os.environ['HF_TOKEN'] = _tok
    HDRS = {'Authorization': f'Bearer {_tok}'} if _tok else {}
    print('token definido para esta sessao.' if _tok else 'nada definido.')

def dl(url, dest_dir, fname):
    if SO_LISTAR:
        print(f'  (listar) {fname} -> {dest_dir.replace(DRIVE_DATA,"")}'); return
    h = HDRS if 'huggingface.co' in url else None
    ok = nbui.download(url, f'{dest_dir}/{fname}', fname, headers=h)
    if not ok and 'huggingface.co' in url and not _tok:
        print('      dica: repo "gated"/privado precisa de token.')
        print('      Configure o secret HF_TOKEN (chave 🔑 no painel esquerdo)')
        print('      ou rode numa celula nova:  pedir_token()  e repita esta celula.')
    return ok

# ---------- 3. resolve ----------
plan, auto, missing = [], [], []
for w in chosen:
    curated = {m['file']: m for m in WFM.get(w['name'], [])}
    for m in curated.values():
        if m.get('optional') and PULAR_OPCIONAIS: continue
        plan.append(m)
    for fn in sorted(wanted_files(w['path'])):
        if fn in curated: continue
        if fn in MODELDB:
            m = MODELDB[fn]
            d = m.get('save_path') or ''
            if d in ('default', '', None):
                d = TYPE2DIR.get(m.get('type',''), 'checkpoints')
            d = d.replace('ComfyUI/models/', '').strip('/')
            plan.append({'file': fn, 'dir': d, 'url': m['url'], 'size': m.get('size','?')})
            auto.append(fn)
        else:
            missing.append((w['name'], fn))

print(f'Espaco livre no Drive: {shutil.disk_usage(DRIVE_DATA).free/1e9:.1f} GB\n')
todo = [m for m in plan]
prog = nbui.Steps(len(todo) or 1, 'Modelos')
seen = set()
for m in plan:
    prog.step(m['file'][:28])
    if m['file'] in seen: continue
    seen.add(m['file'])
    have = already_have(m['file'])
    if have: print(f"  = {m['file']} (ja existe em {have})"); continue
    if m.get('gated') and not _tok:
        print(f"  ! {m['file']} e GATED e nao ha token.")
        print( "      1) aceite as condicoes na pagina do modelo (logado no HF)")
        print( "      2) configure o secret HF_TOKEN (chave no painel esquerdo)")
        continue
    tag = ' [auto]' if m['file'] in auto else ''
    print(f"  + {m['file']} ({m.get('size','?')}){tag}")
    dl(m['url'], f"{DRIVE_DATA}/models/{m['dir']}", m['file'])

prog.close()

if missing:
    print('\n--- SEM download automatico ---')
    for wf, fn in missing:
        note = NOTES.get(fn)
        print(f'  * {fn}  ({wf})')
        if note: print(f'      {note}')
        else:    print('      nao esta na base do Manager. Use Manager > Model Manager,')
        if not note: print('      ou baixe manual e ponha na pasta certa do Drive.')

def hf_normalize(u):
    """Aceita link da pagina do HF e converte para link direto de download.
       .../blob/main/x.safetensors  -> .../resolve/main/x.safetensors"""
    u = u.strip()
    if 'huggingface.co' in u and '/blob/' in u:
        u = u.replace('/blob/', '/resolve/')
    return u

if URL_EXTRA:
    u = hf_normalize(URL_EXTRA)
    if 'huggingface.co' in u and '/resolve/' not in u and '/tree/' not in u:
        print('  ! URL_EXTRA parece ser a pagina do repo, nao um arquivo.')
        print('    Abra a aba "Files", clique no .safetensors e copie o link de download.')
    else:
        dl(u, f'{DRIVE_DATA}/models/{PASTA_EXTRA}', u.split('/')[-1].split('?')[0])

print(f'\nEspaco livre agora: {shutil.disk_usage(DRIVE_DATA).free/1e9:.1f} GB')


In [ ]:
#@title 6. Ligar o ComfyUI { display-mode: "form" }
#@markdown `auto` = deixa o ComfyUI decidir (padrao). Use `lowvram` so se der OOM.
TUNEL = 'colab'  #@param ["colab","cloudflared","ngrok"]
#@markdown `colab` = proxy interno do Google (NAO passa pela internet publica).
#@markdown E o mais rapido; use os outros so se precisar de link externo.
VRAM  = 'auto'         #@param ["auto","highvram","lowvram","novram"]

import subprocess, threading, re, time, os, shlex

NB_VERSION = 'v38-manager'
print('Notebook Celula 6:', NB_VERSION)
print(f'Se NAO aparecer "{NB_VERSION}" acima, voce esta rodando uma copia ANTIGA.')

COMFY='/content/ComfyUI'; DRIVE_DATA='/content/drive/MyDrive/ComfyUI_Data'; PORT=8188

# ================= CHECAGEM DE AMBIENTE (GPU x CPU) =================
def _cmd(c):
    try:
        r = subprocess.run(c, shell=True, capture_output=True, text=True, timeout=30)
        return r.stdout.strip() if r.returncode == 0 else ''
    except Exception:
        return ''

gpu_name = gpu_vram = ''
smi = _cmd('nvidia-smi --query-gpu=name,memory.total,driver_version '
           '--format=csv,noheader,nounits')
if smi:
    parts = [x.strip() for x in smi.split('\n')[0].split(',')]
    if len(parts) >= 2:
        gpu_name, gpu_vram = parts[0], parts[1]

torch_cuda, torch_ver = False, ''
try:
    import torch
    torch_ver  = torch.__version__
    torch_cuda = torch.cuda.is_available()
    if torch_cuda and not gpu_name:
        gpu_name = torch.cuda.get_device_name(0)
except Exception as e:
    torch_ver = f'(torch nao importou: {str(e)[:40]})'

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1e9
except Exception:
    try:
        ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
    except Exception:
        ram_gb = 0

import shutil as _sh
disk_gb = 0
for _p in ('/content', '/'):
    try:
        disk_gb = _sh.disk_usage(_p).free / 1e9; break
    except Exception:
        pass

HAS_GPU = bool(gpu_name) and torch_cuda

print('\n' + '='*64)
print('  AMBIENTE DE EXECUCAO')
print('='*64)
if HAS_GPU:
    print(f'  Acelerador : GPU — {gpu_name}')
    print(f'  VRAM       : {gpu_vram} MB' if gpu_vram else '  VRAM       : ?')
else:
    print('  Acelerador : *** CPU (nenhuma GPU disponivel) ***')
    if gpu_name and not torch_cuda:
        print(f'  Obs        : nvidia-smi ve "{gpu_name}", mas o torch nao acessa CUDA.')
print(f'  PyTorch    : {torch_ver}  | CUDA disponivel: {torch_cuda}')
print(f'  RAM        : {ram_gb:.1f} GB' + (f'   | Disco livre: {disk_gb:.1f} GB' if disk_gb else ''))
print('='*64)

if not HAS_GPU:
    print("""
  ATENCAO: o ComfyUI vai rodar em CPU.

  O que isso significa:
    - Geracao de imagem fica ~20x a 100x mais lenta (minutos por imagem).
    - Workflows 3D (Trellis2) e de rigging (SkinTokens) na pratica NAO rodam:
      dependem de kernels CUDA e vao falhar ao carregar.
    - Modelos grandes (Krea-2, Flux2) provavelmente estouram a RAM.

  Se isso NAO era intencional, troque agora:
    Ambiente de execucao -> Alterar o tipo de ambiente de execucao -> GPU (T4)
    (isso REINICIA a sessao: rode as celulas 1..5 de novo — o que ja esta
     no Drive nao sera baixado outra vez)

  Se foi intencional (so instalar modelos/nodes, testar a UI, editar
  workflows), pode seguir — a interface funciona normalmente.
""")

    resposta = None
    try:
        import ipywidgets as W
        from IPython.display import display
        print('  Ligar o ComfyUI mesmo assim, em modo CPU?')
        _box = W.HBox([
            W.Button(description='Sim, ligar em CPU', button_style='warning'),
            W.Button(description='Nao, vou trocar para GPU', button_style='success')])
        _out = W.Output()
        _ans = {'v': None}
        def _yes(_):
            _ans['v'] = True
            with _out: print('  -> seguindo em CPU...')
        def _no(_):
            _ans['v'] = False
            with _out: print('  -> ok, troque o ambiente e rode a celula de novo.')
        _box.children[0].on_click(_yes); _box.children[1].on_click(_no)
        display(_box, _out)

        # espera a escolha (ate 120s); sem resposta = nao liga
        for _ in range(240):
            if _ans['v'] is not None: break
            time.sleep(0.5)
        resposta = _ans['v']
        if resposta is None:
            print('\n  Sem resposta em 120s — nao vou ligar. Rode a celula de novo.')
    except Exception:
        try:
            resposta = input('  Ligar em CPU mesmo assim? [s/N]: ').strip().lower() in ('s','sim','y','yes')
        except Exception:
            resposta = False

    if not resposta:
        raise SystemExit('Cancelado: troque para GPU (ou confirme CPU) e rode a celula novamente.')

# ---------- Manager novo (pip) ----------
# Nas versoes recentes o Manager virou pacote pip do core; sem isso
# o --enable-manager so imprime warning e a UI nao aparece.
import sys
sys.path.insert(0, '/content/ComfyUI_Colab/config')
import nbui

mreq = f'{COMFY}/manager_requirements.txt'
if os.path.exists(mreq):
    nbui.run(f'pip install -r "{mreq}"', 'deps do ComfyUI-Manager')

# ---------- tunel ----------
LINK = {'url': None}
if TUNEL == 'colab':
    # Proxy interno do Colab: nao sai para a internet publica, entao nao sofre
    # com edge lento do cloudflared. O link so funciona para voce, neste browser.
    try:
        from google.colab import output as _cout
        _url = _cout.eval_js(f'google.colab.kernel.proxyPort({PORT})')
        print('\n' + '='*62)
        print('  LINK DE ACESSO (proxy do Colab):', _url)
        print('  Se nao abrir, use: chave inglesa (painel esquerdo) > Portas > 8188')
        print('='*62 + '\n')
    except Exception as e:
        print('(proxy do Colab indisponivel:', str(e)[:60], ')')
        print('Abra manualmente: chave inglesa no painel esquerdo > Portas > 8188\n')
elif TUNEL == 'cloudflared':
    if not os.path.exists('/usr/local/bin/cloudflared'):
        subprocess.run('wget -q -O /usr/local/bin/cloudflared '
          'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 '
          '&& chmod +x /usr/local/bin/cloudflared', shell=True, check=True)
    def tunnel():
        p = subprocess.Popen(['cloudflared','tunnel','--url',f'http://127.0.0.1:{PORT}'],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            m = re.search(r'https://[-\w]+\.trycloudflare\.com', line)
            if m and not LINK['url']:
                LINK['url'] = m.group(0)
                print('\n' + '='*62)
                print('  LINK DE ACESSO:', LINK['url'])
                print('  Espere aparecer "To see the GUI go to" antes de abrir.')
                print('='*62 + '\n')
    threading.Thread(target=tunnel, daemon=True).start()
    for _ in range(30):
        if LINK['url']: break
        time.sleep(1)
else:
    import getpass
    subprocess.run('pip install -q pyngrok', shell=True, check=True)
    from pyngrok import ngrok
    ngrok.kill(); ngrok.set_auth_token(getpass.getpass('Ngrok authtoken: '))
    print('\nLINK DE ACESSO:', ngrok.connect(PORT,'http').public_url, '\n')

# ---------- flags ----------
# --listen 0.0.0.0 e obrigatorio: com 127.0.0.1 o ComfyUI recusa o Host header
# do tunel (protecao anti DNS-rebinding) e o browser recebe 403.
# ---- user/ LOCAL, espelhado no Drive -------------------------------------
# Servir user/ direto do Drive (FUSE) e a causa da tela "Comfy" travada:
# a UI faz dezenas de leituras pequenas (settings, workflows, cache do
# Manager) e cada uma custa ~100ms no FUSE. Local = instantaneo.
import shutil as _sh
USER_LOCAL = '/content/comfy_user'
DRIVE_USER = f'{DRIVE_DATA}/user'
os.makedirs(DRIVE_USER, exist_ok=True)
os.makedirs(USER_LOCAL, exist_ok=True)
# MERGE sempre (nao "so se nao existir"): a Celula 4 ja criou
# USER_LOCAL/default/workflows antes desta celula rodar. Com o teste antigo,
# o copytree era pulado e settings/__manager nunca chegavam ao local —
# o frontend entao tentava reabrir abas de um settings inexistente e dava
# "Nao foi possivel encontrar o fluxo de trabalho em X.json".
print('Sincronizando user/ do Drive para o disco local...', end=' ', flush=True)
try:
    _sh.copytree(DRIVE_USER, USER_LOCAL, dirs_exist_ok=True)
    print('ok')
except Exception as e:
    print('(parcial:', str(e)[:60], ')')

# os workflows tem de existir nos DOIS lados
_wl = f'{USER_LOCAL}/default/workflows'
_wd = f'{DRIVE_USER}/default/workflows'
for _a, _b in ((_wd, _wl), (_wl, _wd)):
    os.makedirs(_b, exist_ok=True)
    if os.path.isdir(_a):
        for _f in os.listdir(_a):
            if _f.endswith('.json') and not os.path.exists(f'{_b}/{_f}'):
                try: _sh.copy2(f'{_a}/{_f}', f'{_b}/{_f}')
                except Exception: pass
_n = len([f for f in os.listdir(_wl) if f.endswith('.json')])
print(f'Workflows disponiveis para a UI: {_n}')

# ---- SANEIA comfy.settings.json ------------------------------------------
# Causa real do alerta "Nao foi possivel encontrar o fluxo de trabalho em X"
# e das abas travadas: o frontend guarda em Comfy.Workflow.OpenWorkflows a
# lista de abas da sessao anterior. Se UMA entrada aponta para arquivo que
# nao existe mais, a restauracao do tabbar aborta no meio -> so a primeira
# aba abre e os cliques nas outras nao fazem nada.
def _sanear_settings(base):
    """Remove do settings referencias a workflows que nao existem mais.

    IMPORTANTE: isto cobre so o lado SERVIDOR. A restauracao de abas do
    frontend moderno vive no localStorage do NAVEGADOR (draftCacheV2 /
    storageKeys), que o notebook nao alcanca. Por isso a C6 tambem serve
    /destravar_abas.html -- ver adiante.
    """
    p = f'{base}/default/comfy.settings.json'
    if not os.path.exists(p):
        return None
    try:
        st = json.load(open(p, encoding='utf-8'))
    except Exception:
        os.replace(p, p + '.bak')
        json.dump({}, open(p, 'w'))
        return 'settings corrompido -> resetado (backup .bak)'
    wdir = f'{base}/default/workflows'
    existe = set(os.listdir(wdir)) if os.path.isdir(wdir) else set()

    def ok(ref):
        return isinstance(ref, str) and os.path.basename(ref) in existe

    mudou = []
    for _chave in ('Comfy.Workflow.OpenWorkflows', 'Comfy.PreviousWorkflow'):
        _v = st.get(_chave)
        if isinstance(_v, list):
            limpos = [w for w in _v if ok(w)]
            if len(limpos) != len(_v):
                st[_chave] = limpos
                mudou.append(f'{len(_v)-len(limpos)} refs mortas em {_chave}')
        elif _v is not None and not ok(_v):
            st.pop(_chave, None)
            mudou.append(f'{_chave} morto removido')
    _ai = st.get('Comfy.Workflow.ActiveIndex')
    _n = len(st.get('Comfy.Workflow.OpenWorkflows') or [])
    if isinstance(_ai, int) and (_ai < 0 or _ai >= max(_n, 1)):
        st['Comfy.Workflow.ActiveIndex'] = 0 if _n else -1
        mudou.append('ActiveIndex corrigido')
    if mudou:
        json.dump(st, open(p, 'w'), indent=2, ensure_ascii=False)
    return '; '.join(mudou) if mudou else None

for _base in (USER_LOCAL, DRIVE_USER):
    _r = _sanear_settings(_base)
    if _r:
        print(f'  settings ({os.path.basename(_base)}): {_r}')

# ---- Manager: liberar as acoes num ambiente de nuvem -----------------------
# Com --listen 0.0.0.0 o Manager entende que e acesso remoto "publico" e barra
# instalar/atualizar node:
#   "ERROR: To use this action, security_level must be `normal or below`,
#    and network_mode must be set to `personal_cloud`"
# O Colab e exatamente o caso previsto por 'personal_cloud': maquina de nuvem
# de uso pessoal. Sem isto o painel do Manager abre mas nao deixa fazer nada.
try:
    import configparser
    _mp = f'{USER_LOCAL}/__manager'
    os.makedirs(_mp, exist_ok=True)
    _mi = f'{_mp}/config.ini'
    _cp = configparser.ConfigParser()
    if os.path.exists(_mi):
        _cp.read(_mi)
    if not _cp.has_section('default'):
        _cp.add_section('default')
    _cp.set('default', 'network_mode', 'personal_cloud')
    _cp.set('default', 'security_level', 'normal')
    with open(_mi, 'w') as _f:
        _cp.write(_f)
    print('Manager: network_mode=personal_cloud, security_level=normal')
except Exception as e:
    print('(nao consegui ajustar o config.ini do Manager:', str(e)[:60], ')')

# ---- PAGINA DE DESTRAVE (roda no navegador, unico jeito de limpar o estado) --
# A restauracao de abas do frontend vive no localStorage/IndexedDB do NAVEGADOR
# (draftCacheV2). Nenhum arquivo do servidor alcanca isso. Esta pagina e servida
# junto com a UI e limpa o estado do lado do cliente.
_DESTRAVE = r"""<!doctype html><html lang=pt-br><meta charset=utf-8>
<title>Diagnostico / Destrave do ComfyUI</title>
<style>body{font:15px system-ui;max-width:760px;margin:32px auto;padding:0 16px;
background:#1e1e1e;color:#eee}button{font:600 14px system-ui;padding:11px 18px;
margin:5px 4px 5px 0;border:0;border-radius:6px;background:#3b82f6;color:#fff;
cursor:pointer}button:hover{background:#2563eb}pre{background:#111;padding:12px;
border-radius:6px;white-space:pre-wrap;font-size:12.5px;max-height:60vh;
overflow:auto}h1{font-size:19px}.r{background:#dc2626}.g{background:#16a34a}</style>
<h1>Diagnostico / Destrave do ComfyUI</h1>
<p>Se os workflows nao abrem, clique em <b>Testar abertura</b> primeiro: ele
reproduz exatamente o que a UI faz e mostra onde falha.</p>
<button class=g onclick="testar()">Testar abertura</button>
<button onclick="limpar()">Limpar estado das abas</button>
<button class=r onclick="tudo()">Limpar TUDO do site</button>
<pre id=log>Pronto.</pre>
<script>
const log=document.getElementById('log');
const p=(...a)=>{log.textContent+='\n'+a.join(' ');};
async function testar(){
  log.textContent='Testando igual a UI faz...\n';
  try{
    const u='./api/userdata?dir=workflows&recurse=true&split=false&full_info=true';
    const r=await fetch(u);
    p('1) listar workflows -> HTTP',r.status);
    if(!r.ok){p('   !! a listagem falhou. Corpo:',(await r.text()).slice(0,300));return;}
    const lista=await r.json();
    p('   '+lista.length+' workflow(s):');
    lista.forEach(x=>p('     -',x.path||x));
    if(!lista.length){p('   !! lista vazia');return;}
    for(const it of lista){
      const nome=it.path||it;
      const enc=encodeURIComponent('workflows/'+nome);
      const g=await fetch('./api/userdata/'+enc);
      if(!g.ok){p('2) GET',nome,'-> HTTP',g.status,'FALHOU');continue;}
      const txt=await g.text();
      let d;
      try{d=JSON.parse(txt);}catch(e){
        p('2) GET',nome,'-> JSON INVALIDO:',e.message);continue;}
      const nos=(d.nodes||[]).length;
      const tipos=[...new Set((d.nodes||[]).map(n=>n.type))];
      p('2) GET',nome,'->',txt.length,'bytes,',nos,'nos  OK');
      const oi=await (await fetch('./api/object_info')).json();
      const faltam=tipos.filter(t=>t!=='Note'&&t!=='MarkdownNote'&&!(t in oi));
      if(faltam.length)p('   !! NOS AUSENTES NO SERVIDOR:',faltam.join(', '));
      else p('   todos os',tipos.length,'tipos de no existem no servidor');
    }
    p('\nSe tudo acima esta OK, o problema e o estado do navegador:');
    p('clique em "Limpar estado das abas".');
  }catch(e){p('ERRO:',e.message);}
}
function limpar(){
  let alvos=[];
  for(let i=0;i<localStorage.length;i++){
    const k=localStorage.key(i);
    if(/workflow|draft|tab|Comfy\.Workflow|litegraph/i.test(k)) alvos.push(k);
  }
  alvos.forEach(k=>localStorage.removeItem(k));
  const req=indexedDB.deleteDatabase('comfy-workflow-drafts');
  req.onsuccess=req.onerror=req.onblocked=()=>{
    log.textContent='Removidas '+alvos.length+' chave(s):\n'+alvos.join('\n')+
      '\n\nRecarregando...';
    setTimeout(()=>location.href='./',1200);
  };
}
function tudo(){
  localStorage.clear(); sessionStorage.clear();
  const fim=()=>{log.textContent='Tudo limpo. Recarregando...';
    setTimeout(()=>location.href='./',1200);};
  indexedDB.databases?indexedDB.databases().then(ds=>{
    ds.forEach(d=>indexedDB.deleteDatabase(d.name));fim();}):fim();
}
</script></html>"""
# O web_root NAO e ComfyUI/web/ nas versoes novas: vem do pacote pip
# comfyui_frontend_package (server.py: FrontendManager.init_frontend).
# Escrever em ComfyUI/web/ nao seria servido. Descobrimos o caminho real:
_alvos = []
try:
    import comfyui_frontend_package as _fp
    _alvos.append(os.path.join(os.path.dirname(_fp.__file__), 'static'))
except Exception:
    pass
_alvos.append(f'{COMFY}/web')                      # fallback versoes antigas
_ok = None
for _r in _alvos:
    try:
        if not os.path.isdir(_r):
            continue
        open(f'{_r}/destravar.html', 'w', encoding='utf-8').write(_DESTRAVE)
        _ok = _r
        break
    except Exception:
        continue
if _ok:
    print('DESTRAVAR ABAS -> abra  <URL_DA_UI>/destravar.html')
else:
    print('(destravar.html nao pode ser instalado; use o Clear site data do navegador)')

def _sync_user_to_drive(intervalo=120):
    """Salva user/ de volta no Drive periodicamente (workflows, settings)."""
    while True:
        time.sleep(intervalo)
        try:
            _sh.copytree(USER_LOCAL, DRIVE_USER, dirs_exist_ok=True)
        except Exception:
            pass
threading.Thread(target=_sync_user_to_drive, daemon=True).start()
print(f'user/ local: {USER_LOCAL}  (sincroniza no Drive a cada 2 min)')

def salvar_agora():
    """Forca o backup do user/ para o Drive. Rode antes de encerrar a sessao."""
    _sh.copytree(USER_LOCAL, DRIVE_USER, dirs_exist_ok=True)
    print('user/ salvo no Drive.')

# Flags de memoria. O gargalo desta maquina e RAM DE CPU (~13 GB), nao VRAM.
# Sintoma de falta de RAM: o processo MORRE/trava sem mensagem. Falta de VRAM
# aparece como erro CUDA/torch. Video e animacao estouram a RAM primeiro.
#   --cache-none          descarrega cada modelo apos usar (o mais eficaz)
#   --disable-smart-memory  nao segura modelo na memoria "por precaucao"
#   --mmap-torch-files    le o peso do disco em vez de copiar tudo para a RAM
ECONOMIZAR_RAM = True  # desligue se for so gerar imagem e quiser mais velocidade

args = ['--listen','0.0.0.0','--port',str(PORT),
        '--enable-cors-header','*',
        '--output-directory', f'{DRIVE_DATA}/output',
        '--input-directory',  f'{DRIVE_DATA}/input',
        '--user-directory',   USER_LOCAL,
        '--preview-method','auto','--disable-auto-launch']
if ECONOMIZAR_RAM:
    args += ['--cache-none','--disable-smart-memory','--mmap-torch-files']

if not HAS_GPU:
    args.append('--cpu')          # sem isso o ComfyUI tenta CUDA e quebra
elif VRAM != 'auto':
    args.append(f'--{VRAM}')

try:
    helptxt = subprocess.run(['python','main.py','--help'], cwd=COMFY,
                             capture_output=True, text=True, timeout=180).stdout
    args = [a for a in args if not (a.startswith('--') and a not in helptxt)]
    if '--enable-manager' in helptxt:
        args.append('--enable-manager')
except Exception as e:
    print('(nao consegui checar --help:', e, ')')

# ---- AUTODIAGNOSTICO -----------------------------------------------------
# A celula abaixo (main.py) BLOQUEIA, entao a medicao roda numa thread:
# ela espera o servidor subir e imprime os tempos no meio deste mesmo log.
def _diag_workflows():
    """Compara o DISCO com o que a API /userdata devolve.

    A sidebar NAO le o disco: ela chama GET /api/userdata?dir=workflows.
    Espera a porta abrir (o boot pode levar minutos importando custom nodes)
    em vez de dormir um tempo fixo.
    """
    import urllib.request, urllib.parse, socket
    base = f'http://127.0.0.1:{PORT}'
    # --- espera a porta REALMENTE aceitar conexao (ate 5 min) ---
    for _ in range(150):
        try:
            with socket.create_connection(('127.0.0.1', PORT), timeout=2):
                break
        except OSError:
            time.sleep(2)
    else:
        print('\n[diag] servidor nao subiu em 5 min; diagnostico cancelado\n')
        return
    time.sleep(3)
    print('\n===== DIAGNOSTICO DOS WORKFLOWS =====')
    wdir = f'{USER_LOCAL}/default/workflows'
    disco = sorted(f for f in os.listdir(wdir)) if os.path.isdir(wdir) else []
    print(f'DISCO ({wdir}): {len(disco)} arquivo(s)')
    for f in disco:
        print('   ', f)
    api = None
    # ESTA e a chamada exata que a sidebar faz (api.listUserDataFullInfo):
    #   /userdata?dir=workflows&recurse=true&split=false&full_info=true
    for rota in ('/api/userdata?dir=workflows&recurse=true&split=false&full_info=true',
                 '/api/userdata?dir=workflows'):
        try:
            with urllib.request.urlopen(base + rota, timeout=30) as r:
                corpo = r.read()
            api = json.loads(corpo)
            print(f'API ({rota.split("?")[0]}) -> HTTP 200, {len(corpo)} bytes, '
                  f'{len(api)} item(s)')
            print(f'   conteudo bruto: {corpo[:400].decode("utf-8", "replace")}')
            break
        except Exception as e:
            print(f'   rota {rota.split("?")[0]} falhou: {str(e)[:70]}')
    if api is not None:
        nomes = {os.path.basename(x['path'] if isinstance(x, dict) else str(x))
                 for x in api}
        faltam = [f for f in disco if f not in nomes]
        if faltam:
            print(f'   !! NO DISCO MAS NAO NA API: {faltam}')
            print('      -> o SERVIDOR nao enxerga estes arquivos.')
        elif not disco:
            print('   !! o diretorio de workflows esta VAZIO no disco.')
        else:
            print('   OK: a API devolve tudo que esta no disco.')
            print('      -> arquivos sadios. O estado quebrado esta no NAVEGADOR:')
            print('         abra  <URL_DA_UI>/destravar.html')
    if disco:
        alvo = urllib.parse.quote(f'workflows/{disco[0]}', safe='')
        try:
            with urllib.request.urlopen(f'{base}/api/userdata/{alvo}', timeout=30) as r:
                corpo = r.read()
            d = json.loads(corpo)
            print(f'GET {disco[0]}: {len(corpo)} bytes, {len(d.get("nodes", []))} nos '
                  f'-> ABRIVEL')
        except Exception as e:
            print(f'GET {disco[0]} FALHOU: {str(e)[:90]}')
            print('   -> e AQUI que a UI quebra.')
    print('=====================================\n')

threading.Thread(target=_diag_workflows, daemon=True).start()

def _autodiag():
    import urllib.request
    base = f'http://127.0.0.1:{PORT}'
    for _ in range(600):                      # espera ate 10 min pelo boot
        time.sleep(1)
        try:
            urllib.request.urlopen(base + '/system_stats', timeout=5).read()
            break
        except Exception:
            continue
    else:
        return
    eps = ['/system_stats', '/queue', '/api/userdata?dir=workflows',
           '/embeddings', '/object_info']
    linhas, total = [], 0.0
    for ep in eps:
        t0 = time.time()
        try:
            with urllib.request.urlopen(base + ep, timeout=300) as r:
                n = len(r.read())
            dt = time.time() - t0
            linhas.append(f'  {ep:30} {dt:7.2f}s  {n/1024:8.0f} KB'
                          + ('   <<< LENTO' if dt > 3 else ''))
        except Exception as e:
            dt = time.time() - t0
            linhas.append(f'  {ep:30} {dt:7.2f}s  ERRO {type(e).__name__}')
        total += dt
    print('\n' + '='*66)
    print('  DIAGNOSTICO DA UI (medido no localhost, sem o tunel)')
    print('='*66)
    for l in linhas: print(l)
    print(f'  {"TOTAL":30} {total:7.2f}s')
    lentos = [l for l in linhas if 'LENTO' in l or 'ERRO' in l]
    if not lentos and total < 5:
        print('  >> Servidor RAPIDO. O gargalo esta no TUNEL ou no navegador.')
        print('     Use o proxy do Colab: chave inglesa > Portas > 8188.')
    else:
        print('  >> Ha endpoint(s) lento(s) acima - o gargalo e o SERVIDOR.')
        print('     /object_info lento = varredura das pastas de modelos.')
        print('     Reduza custom nodes ativos (Celula 4) e arquivos soltos.')
    print('='*66 + '\n')
threading.Thread(target=_autodiag, daemon=True).start()

print('$ python main.py ' + ' '.join(shlex.quote(a) for a in args) + '\n')
!cd {COMFY} && python main.py {' '.join(shlex.quote(a) for a in args)}


## Sobre VRAM e abrir tudo junto

Os 6 GB são o pico de **um** workflow rodando. O problema de abrir os três juntos não é
o pico — é que o ComfyUI mantém em VRAM o último modelo carregado de cada execução, e
aba aberta com workflow grande também custa RAM de CPU. Misturar SDXL + AnimateDiff +
3D na mesma sessão faz o Colab começar a fazer swap e, no T4 (16 GB), OOM.

Por isso a Célula 3: uma sessão = um propósito.

- **Um workflow por sessão** é o ideal.
- Precisa alternar? Use **Free model and node cache** no menu do ComfyUI antes de trocar.
- `lowvram` na Célula 6 se estourar; `highvram` só em A100.

## Como funciona a seleção

1. Salve os workflows em `ComfyUI_Data/workflows/` (pode usar subpastas: `splash/`, `3d/`, `anim/`).
2. Célula 3 lê o JSON de cada um, extrai os `class_type` e cruza com o `class_map` do registry.
3. Célula 4 clona só os pacotes daqueles workflows, e renomeia o resto para `.disabled`
   — não apaga nada, e reativar é só marcar o workflow de novo.
4. Os repos ficam em cache em `ComfyUI_Data/node_cache/`, então a segunda sessão não baixa nada.

## Quando aparecer um node desconhecido

Se um workflow usa um node que não está no `class_map`, ele não é instalado e o ComfyUI
mostra "missing node". Aí: **Manager → Install Missing Custom Nodes**, veja o nome do pacote,
e adicione o par `"NomeDoClassType": "NomeDoPacote"` em `config/node_registry.json`
(mais o repo em `packs`, se for novo). Na próxima sessão ele entra sozinho.

**Me manda os três workflows** que eu preencho o registry com os nodes reais deles e testo o parser.
